# 第67章 漏斗图（px.funnel）

用漏斗宽度展示流程各阶段人数和转化损失。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

业务流程具有明确先后阶段，需要查看流失。

## 数据结构

有序阶段列和人数或数量列。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 修改 textinfo 从 "value+percent initial+percent previous" 为 "value+percent previous"，对比整体转化率与相邻阶段转化率
2. 在分渠道漏斗中添加 hovertemplate 自定义悬停信息格式，查看各阶段精确人数
3. 将 color 映射为转化率指标，说明颜色编码对定位最大流失环节的作用


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

funnel = pd.DataFrame({
    "stage": ["访问", "查看商品", "加入购物车", "提交订单", "支付成功"],
    "users": [12000, 7200, 3100, 1850, 1420],
})
timeline = pd.DataFrame({
    "task": ["数据准备", "探索分析", "图表制作", "报告复核"],
    "start": pd.to_datetime(["2026-03-01", "2026-03-04", "2026-03-08", "2026-03-12"]),
    "finish": pd.to_datetime(["2026-03-04", "2026-03-09", "2026-03-13", "2026-03-15"]),
    "owner": ["数据", "分析", "分析", "负责人"],
})
from js import window
base_url = window.location.origin
diamonds = pd.read_csv(f"{base_url}/datasets/diamonds.csv")
orders_full = diamonds.assign(
    date=pd.Timestamp("2026-01-01"), category=diamonds["cut"], region=diamonds["clarity"],
    channel=diamonds["color"], order_value=diamonds["price"], items=diamonds["carat"],
    sales=diamonds["price"], month="公开样本",
)
orders = orders_full.sample(5_000, random_state=55).copy()
flights = pd.read_csv(f"{base_url}/datasets/flights.csv")
monthly = flights.query("year == 1960").rename(columns={"passengers": "sales"}).copy()
monthly["orders"] = monthly["sales"]
monthly["profit"] = monthly["sales"].rolling(3, min_periods=1).mean()
regional = orders_full.groupby(["region", "channel"], as_index=False)["sales"].sum()
hierarchy = diamonds.groupby(["cut", "color"], as_index=False)["price"].sum().rename(
    columns={"cut": "department", "color": "category", "price": "sales"}
)
gapminder = pd.read_csv(f"{base_url}/datasets/gapminder.csv")
countries = gapminder.query("year == 2007").assign(
    country=lambda frame: frame["country"], market=lambda frame: frame["country"],
    sales=lambda frame: frame["gdpPercap"], growth=lambda frame: frame["lifeExp"]
)
print(f"Diamonds：{len(diamonds):,} 行；Flights：{len(flights):,} 行；Gapminder：{len(gapminder):,} 行")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig = px.funnel(funnel, x="users", y="stage", title="用户转化漏斗")
fig.update_traces(textinfo="value+percent initial+percent previous")
fig.update_layout(xaxis_title="用户数", yaxis_title="阶段")
fig.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
funnel_detail = pd.DataFrame({
    "stage": funnel["stage"].tolist() * 2,
    "channel": ["自然流量"] * len(funnel) + ["广告"] * len(funnel),
    "users": [7000, 4500, 2100, 1280, 1010, 5000, 2700, 1000, 570, 410],
})
fig = px.funnel(funnel_detail, x="users", y="stage", color="channel", title="分渠道转化漏斗")
fig.update_layout(xaxis_title="用户数", yaxis_title="阶段", legend_title="渠道")
fig.show()


## 3. 参数说明

- orientation：方向
- textinfo：标签
- funnelmode：分组
- category_orders：阶段顺序


## 4. 结果解读

计算阶段转化率和累计转化率，重点定位最大流失环节。


## 常见误区

- 阶段不是同一批用户
- 忽略时间窗口
- 只看绝对流失不看转化率


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
practice_funnel = funnel.copy()
practice_funnel["previous_rate"] = practice_funnel["users"] / practice_funnel["users"].shift(1)
print(practice_funnel)
fig = px.funnel(practice_funnel, x="users", y="stage", color="previous_rate", color_continuous_scale="Blues", title="转化率着色漏斗")
fig.show()


## 本章小结

用漏斗宽度展示流程各阶段人数和转化损失。


### 你已经掌握

- 判断漏斗图（px.funnel）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | 业务流程具有明确先后阶段，需要查看流失。 |
| 数据结构 | 有序阶段列和人数或数量列。 |
| 结果解读 | 计算阶段转化率和累计转化率，重点定位最大流失环节。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `orientation` | 方向 |
| `textinfo` | 标签 |
| `funnelmode` | 分组 |
| `category_orders` | 阶段顺序 |


### 需要注意

- 阶段不是同一批用户
- 忽略时间窗口
- 只看绝对流失不看转化率


### 完成检查

- [ ] 能判断什么问题适合使用漏斗图（px.funnel）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
